[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance4_correction.ipynb)

# Séance 4.4 — Segmenter sans étiquette — quatre clients, quatre traitements

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer un problème supervisé d'un problème non supervisé
- expliquer pourquoi il faut standardiser avant de mesurer une distance
- appliquer `KMeans` et choisir le nombre de groupes
- donner un nom et un chiffre d'affaires à chaque segment obtenu
- transformer une segmentation en plan d'action budgété

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cli = pd.read_csv(BASE + "clients_rfm.csv")

variables = ["recence", "freq", "montant"]
Xs = StandardScaler().fit_transform(np.log1p(cli[variables]))
print(cli.shape, "| variables mises a l'echelle")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Les trois variables

> **Votre mission :**
> - Afficher le `describe()` des trois variables RFM.
> - Mettre le montant maximum dans `max_montant` et la récence maximum dans `max_recence`.
> - Le rapport entre les deux explique pourquoi il faut standardiser.

In [ ]:
print(cli[variables].describe().round(1))

max_montant = cli["montant"].max()
max_recence = cli["recence"].max()
print(max_montant, "euros contre", max_recence, "jours")

# Un facteur 400 entre les deux echelles : sans correction, le montant
# ecrasera tout dans le calcul des distances.

In [ ]:
verifier("1a - montant maximum", abs(max_montant - 143825.06) < 1, "max() sur montant")
verifier("1b - recence maximum", max_recence == 372, "max() sur recence")

### Exercice 2 — Le clustering qui rate

> **Votre mission :**
> - Lancer un `KMeans` à 4 groupes sur les colonnes **brutes**, sans standardiser → `brut`.
> - Mettre la taille du plus gros groupe dans `plus_gros` et celle du plus petit dans `plus_petit`.

In [ ]:
brut = KMeans(n_clusters=4, n_init=10, random_state=42).fit(cli[variables])
tailles = pd.Series(brut.labels_).value_counts()

plus_gros = tailles.max()
plus_petit = tailles.min()
print(plus_gros, "clients dans le plus gros groupe,", plus_petit, "dans le plus petit")

# 401 contre 2 : ce decoupage ne segmente rien du tout.

In [ ]:
verifier("2a - plus gros groupe", plus_gros == 401, "value_counts() puis max()")
verifier("2b - plus petit groupe", plus_petit == 2, "la methode est min()")

### Exercice 3 — Standardiser

> **Votre mission :**
> - `Xs` est préparé dans la cellule de préparation : `log1p` puis `StandardScaler`.
> - Vérifier le résultat : mettre la moyenne de la première colonne dans `moy0` et son écart-type dans `ec0`, arrondis à 2 décimales.

In [ ]:
moy0 = round(Xs[:, 0].mean(), 2)
ec0 = round(Xs[:, 0].std(), 2)

print("moyenne", moy0, "| ecart-type", ec0)

# C'est ce que fait StandardScaler : moyenne 0, ecart-type 1, pour que
# chaque variable pese pareil dans le calcul des distances.

In [ ]:
verifier("3a - moyenne apres mise a l'echelle", moy0 == 0.0, "mean() sur la premiere colonne")
verifier("3b - ecart-type", ec0 == 1.0, "la methode est std()")

### Exercice 4 — La courbe du coude

> **Votre mission :**
> - Pour `k` de 2 à 8, relever l'inertie du `KMeans` → `inerties` (une `Series` indexée par k).
> - La tracer. Mettre l'inertie à `k` = 4 dans `inertie4`, arrondie à 1 décimale.

In [ ]:
inerties = pd.Series(
    {k: KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).inertia_
     for k in range(2, 9)})

inerties.plot(marker="o", figsize=(7, 4))
plt.title("Courbe du coude")
plt.show()

inertie4 = round(inerties[4], 1)
print(inertie4)

In [ ]:
verifier("4 - inertie a k=4", abs(inertie4 - 453.7) < 5,
         "l'attribut s'appelle inertia_, avec un underscore final")

### Exercice 5 — La silhouette

> **Votre mission :**
> - Calculer le score de silhouette pour `k` de 2 à 6 → `silhouettes`.
> - Mettre le `k` qui la maximise dans `k_silhouette`.
> - Est-ce celui qu'on va retenir ?

In [ ]:
silhouettes = pd.Series(
    {k: silhouette_score(Xs, KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).labels_)
     for k in range(2, 7)})

k_silhouette = silhouettes.idxmax()
print(silhouettes.round(3))
print("meilleur k selon la silhouette :", k_silhouette)

# Elle prefere 2. On retiendra pourtant 4 : deux groupes, ce sont "les
# bons" et "les autres", et aucune action ne s'en deduit.

In [ ]:
verifier("5 - k preferee par la silhouette", k_silhouette == 2,
         "idxmax() donne l'indice du maximum")

### Exercice 6 — Former les quatre groupes

> **Votre mission :**
> - Ajuster un `KMeans` à 4 groupes sur `Xs` → `km`, et ranger les étiquettes dans une colonne `groupe` de `cli`.
> - Mettre la taille du plus petit groupe dans `taille_min`.

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
cli["groupe"] = km.labels_

taille_min = cli["groupe"].value_counts().min()
print(cli["groupe"].value_counts().sort_index())

# 69, 98, 149 et 156 : quatre groupes de taille comparable, exploitables.

In [ ]:
verifier("6 - plus petit groupe", taille_min == 69,
         "l'attribut des etiquettes s'appelle labels_")

### Exercice 7 — Le tableau des personas

> **Votre mission :**
> - Construire `profils` : par groupe, l'effectif, les **médianes** de récence, fréquence et montant, et la somme des montants.
> - Ajouter une colonne `part_ca` en % du CA total.

In [ ]:
profils = cli.groupby("groupe").agg(
    n=("client_id", "size"),
    recence=("recence", "median"),      # mediane : bloc 3, seance 3.1
    freq=("freq", "median"),
    montant=("montant", "median"),
    ca=("montant", "sum"),              # somme ici : on veut le CA total
)
profils["part_ca"] = (100 * profils["ca"] / cli["montant"].sum()).round(1)
print(profils.round(1))

In [ ]:
verifier("7 - part du CA du plus gros segment", abs(profils["part_ca"].max() - 60.7) < 1,
         "median() pour les profils, sum() pour le CA")

### Exercice 8 — Nommer les groupes

> **Votre mission :**
> - Identifier les groupes par leurs propriétés, sans se fier au numéro (il change d'une exécution à l'autre).
> - Le groupe des **dormants** est celui dont la récence médiane est la plus élevée → `g_dormants`.
> - Celui des **champions** est celui dont la fréquence médiane est la plus élevée → `g_champions`.

In [ ]:
# idxmax() renvoie l'etiquette de la ligne, donc le numero du groupe
g_dormants = profils["recence"].idxmax()
g_champions = profils["freq"].idxmax()

print("dormants : groupe", g_dormants, "| champions : groupe", g_champions)

# On identifie par le COMPORTEMENT, jamais par le numero : k-means
# n'attribue pas ses etiquettes dans un ordre stable.

In [ ]:
verifier("8a - segment dormant", profils.loc[g_dormants, "recence"] > 150,
         "la recence la plus elevee : idxmax()")
verifier("8b - segment champion", profils.loc[g_champions, "freq"] >= 9,
         "la frequence la plus elevee")

### Exercice 9 — Le nuage coloré

> **Votre mission :**
> - Tracer récence en abscisse et fréquence en ordonnée, un point par client, une couleur par groupe.
> - Mettre la fréquence en **échelle logarithmique** — sinon les champions écrasent la figure.
> - Mettre le nombre de clients tracés dans `nb_points`.

In [ ]:
for g in sorted(cli["groupe"].unique()):
    part = cli.query("groupe == @g")
    plt.scatter(part["recence"], part["freq"], alpha=0.6, label=f"groupe {g}")

# yscale("log") : sans elle, le client a 201 commandes ecrase les 471 autres
plt.yscale("log")
plt.xlabel("jours depuis le dernier achat")
plt.ylabel("nombre de commandes (echelle log)")
plt.legend()
plt.show()

nb_points = len(cli)

In [ ]:
verifier("9 - tous les clients sont traces", nb_points == 472,
         "la methode s'appelle plt.yscale")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Chiffrer l'arbitrage : le CA déjà dépensé par les dormants → `ca_dormants` (2 décimales), et leur nombre → `nb_dormants`.
> - Puis le nombre de clients du segment « nouveaux » — celui dont la fréquence est la plus basse **et** la récence faible → `nb_nouveaux`.
> - Où mettriez-vous 5 000 € de relance ? Répondez en commentaire.

In [ ]:
dormants = cli.query("groupe == @g_dormants")
ca_dormants = round(dormants["montant"].sum(), 2)
nb_dormants = len(dormants)

# Les "nouveaux" : peu de commandes, mais un achat recent
recents = profils.query("recence < 100")
g_nouveaux = recents["freq"].idxmin()
nb_nouveaux = int(profils.loc[g_nouveaux, "n"])
print(nb_dormants, "dormants (", ca_dormants, "euros ) |", nb_nouveaux, "nouveaux")

# Une reponse possible :
# "Je mets le budget sur les 98 nouveaux. Les dormants n'ont commande
#  qu'une fois et se taisent depuis six mois : leur 74 683 EUR est un
#  historique, pas un potentiel. Les nouveaux viennent d'acheter, ils sont
#  encore attentifs, et le passage a la deuxieme commande est le moment
#  ou un acheteur devient client. A verifier par un test — bloc 5."

In [ ]:
verifier("10a - nombre de dormants", nb_dormants == 149, "sum() sur les montants du segment")
verifier("10b - CA des dormants", abs(ca_dormants - 74683.3) < 100, "somme des montants")
verifier("10c - nombre de nouveaux", nb_nouveaux == 98, "la frequence la plus basse parmi les recents")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — La segmentation est-elle stable ?

> **Votre mission :**
> - Relancer `KMeans` avec cinq `random_state` différents et comparer les tailles de groupes obtenues.
> - Les segments sont-ils les mêmes ? Que faudrait-il vérifier avant de bâtir une campagne dessus ?

In [ ]:
for graine in range(5):
    k = KMeans(n_clusters=4, n_init=10, random_state=graine).fit(Xs)
    print(f"graine {graine} :", sorted(pd.Series(k.labels_).value_counts().tolist()))

# Les tailles sont identiques d'une graine a l'autre : la structure est
# stable, seuls les NUMEROS changent. C'est le controle a faire avant de
# batir quoi que ce soit dessus — une segmentation instable ne serait
# qu'un artefact du tirage initial.

### Question 12 — Segmenter sans le montant

> **Votre mission :**
> - Refaire la segmentation sur `recence` et `freq` seulement.
> - Comparer les profils obtenus à ceux du cours.
> - Le montant apportait-il quelque chose que les deux autres variables ne disaient pas ?

In [ ]:
X2 = StandardScaler().fit_transform(np.log1p(cli[["recence", "freq"]]))
k2 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X2)

cli.assign(g2=k2.labels_).groupby("g2").agg(
    n=("client_id", "size"), recence=("recence", "median"),
    freq=("freq", "median"), montant=("montant", "median")).round(1)

# Les groupes se ressemblent beaucoup : le montant est tres correle a la
# frequence (0,87, mesure en seance 3.3). Une variable redondante
# n'apporte pas d'information, elle apporte du poids — ici, elle comptait
# deux fois la meme chose.

### Question 13 — Le segment qui pèse

> **Votre mission :**
> - Pour chaque groupe, calculer la part du CA **et** la part des effectifs, puis leur rapport.
> - Un rapport supérieur à 1 signale un segment qui pèse plus que son nombre.

In [ ]:
poids = cli.groupby("groupe").agg(n=("client_id", "size"), ca=("montant", "sum"))
poids["part_clients"] = (100 * poids["n"] / len(cli)).round(1)
poids["part_ca"] = (100 * poids["ca"] / cli["montant"].sum()).round(1)
poids["rapport"] = (poids["part_ca"] / poids["part_clients"]).round(2)

poids.sort_values("rapport", ascending=False).round(1)

# Les champions : 15 % des clients, 61 % du CA — un rapport de 4. Les
# dormants font l'inverse. C'est ce rapport, et non le CA brut, qui dit
# ou un euro de marketing rapporte le plus.

### Question 14 — Segmenter les pays plutôt que les clients

> **Votre mission :**
> - Croiser les groupes obtenus avec la colonne `pays`, en pourcentage de ligne.
> - Un marché a-t-il un profil de clientèle particulier ?
> - *Rappel :* `pd.crosstab(a, b, normalize='index')`, vu en séance 3.3.

In [ ]:
gros = cli["pays"].value_counts()
gros = gros[gros >= 15].index

sub = cli.query("pays in @gros")
(pd.crosstab(sub["pays"], sub["groupe"], normalize="index") * 100).round(1)

# La composition varie nettement d'un marche a l'autre. Une campagne
# uniforme sur tous les pays traiterait donc des clienteles differentes
# de la meme facon.

### Question 15 — Le client le plus atypique

> **Votre mission :**
> - Pour chaque client, calculer sa distance au centre de son propre groupe.
> - Afficher les cinq plus éloignés. Que sont-ils ?
> - *Nouveau :* `km.transform(Xs)` donne la distance de chaque point à **chaque** centre.

In [ ]:
distances = km.transform(Xs)
cli["distance"] = distances[np.arange(len(cli)), km.labels_]

cli.nlargest(5, "distance")[["client_id", "recence", "freq", "montant",
                             "groupe", "distance"]].round(2)

# Ce sont les clients que la segmentation decrit le plus mal : des profils
# qui ne ressemblent a aucun des quatre types. Sur un vrai projet, ce sont
# eux qu'on va regarder a la main — ils revelent souvent un cinquieme
# segment qu'on n'avait pas vu.

### Question 16 — Segmenter les produits

> **Votre mission :**
> - Charger `produits_profil.csv` : 1 263 références avec leur profil de vente.
> - Appliquer la même méthode — `log1p`, standardisation, `KMeans` à 4 groupes — sur `nb_cmd`, `prix`, `pays` et `part_q4`.
> - Décrire les quatre familles obtenues.

In [ ]:
prod = pd.read_csv(BASE + "produits_profil.csv")
vp = ["nb_cmd", "prix", "pays", "part_q4"]

Xp = StandardScaler().fit_transform(np.log1p(prod[vp]))
kp = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xp)
prod["famille"] = kp.labels_

prod.groupby("famille").agg(
    n=("prod_id", "size"), nb_cmd=("nb_cmd", "median"), prix=("prix", "median"),
    pays=("pays", "median"), part_q4=("part_q4", "median"), ca=("ca", "sum")).round(2)

### Question 17 — La famille saisonnière

> **Votre mission :**
> - Parmi les familles de produits, identifier celle dont la part du chiffre d'affaires réalisée au dernier trimestre est la plus élevée.
> - Combien de références contient-elle, et quel CA représente-t-elle ?
> - Quelle décision d'approvisionnement en tirez-vous ?

In [ ]:
par_famille = prod.groupby("famille").agg(
    n=("prod_id", "size"), part_q4=("part_q4", "median"), ca=("ca", "sum"))

saison = par_famille["part_q4"].idxmax()
print("famille la plus saisonniere :", saison)
print(par_famille.loc[saison].round(2))

# Ces references concentrent leurs ventes sur trois mois. Elles doivent
# etre stockees avant octobre et ne pas encombrer l'entrepot le reste de
# l'annee — une decision qu'aucun classement par chiffre d'affaires
# n'aurait fait apparaitre.

### Question 18 — Le bon nombre de familles

> **Votre mission :**
> - Tracer la courbe du coude et celle de la silhouette pour les produits, de `k` = 2 à 8.
> - Les deux indicateurs s'accordent-ils ? Que retenez-vous, et pourquoi ?

In [ ]:
lignes = []
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xp)
    lignes.append({"k": k, "inertie": m.inertia_,
                   "silhouette": silhouette_score(Xp, m.labels_)})

mesures = pd.DataFrame(lignes).set_index("k")
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
mesures["inertie"].plot(ax=ax[0], marker="o", title="coude")
mesures["silhouette"].plot(ax=ax[1], marker="o", title="silhouette")
plt.tight_layout()
plt.show()

print(mesures.round(3))

### Question 19 — Croiser les deux segmentations

> **Votre mission :**
> - Question ouverte, et plus difficile : les champions achètent-ils les mêmes familles de produits que les autres segments ?
> - Il faut repartir de `ventes.csv` du bloc 2 pour relier clients et produits. L'URL est donnée ci-dessous.
> - *Indice :* joindre les ventes aux groupes clients, puis aux familles produits, puis croiser.

In [ ]:
base2 = BASE.replace("bloc4_ml", "bloc2_donnees")
ventes = pd.read_csv(base2 + "ventes.csv")

v = (ventes.merge(cli[["client_id", "groupe"]], on="client_id")
           .merge(prod[["prod_id", "famille"]], on="prod_id"))

(pd.crosstab(v["groupe"], v["famille"], normalize="index") * 100).round(1)

# Les profils d'achat different d'un segment a l'autre : les champions ne
# consomment pas le meme catalogue que les acheteurs occasionnels. Cela
# ouvre une recommandation produit par segment — et c'est le genre de
# croisement qu'aucune des deux segmentations ne montrait separement.

### Question 20 — Le livrable du bloc

> **Votre mission :**
> - Rédigez la note qui clôt le bloc 4 : *« comment devons-nous traiter nos clients ? »*
> - Contrainte : quatre segments nommés, chacun avec son effectif, sa part du CA et **une** action ; plus une phrase sur ce que ces données ne permettent pas d'affirmer.
> - Calculez d'abord le tableau dont vous avez besoin.

In [ ]:
note = cli.groupby("groupe").agg(
    clients=("client_id", "size"),
    recence=("recence", "median"),
    freq=("freq", "median"),
    ca=("montant", "sum"),
)
note["part_ca"] = (100 * note["ca"] / cli["montant"].sum()).round(1)
note.sort_values("part_ca", ascending=False).round(1)

# Note possible :
# "Nos 472 clients se repartissent en quatre profils nets.
#  - Champions (69 clients, 61 % du CA) : commandent tous les mois. Action
#    = securiser. Un contact nomme, des le premier signe de ralentissement.
#  - Fideles (156, 30 %) : reguliers sans etre intensifs. Action = faire
#    monter en frequence.
#  - Nouveaux (98, 3 %) : une seule commande, recente. Action = obtenir la
#    deuxieme, c'est la qui se joue la fidelisation.
#  - Dormants (149, 6 %) : six mois de silence. Action = une relance unique
#    et peu couteuse, puis on arrete d'investir.
#  Limite : cette segmentation DECRIT ce qui est, elle ne prouve pas
#  qu'une action changera quoi que ce soit. Aucun des chiffres ci-dessus
#  ne dit qu'une relance fonctionne — il faut la tester sur un groupe et
#  pas sur l'autre. C'est l'objet du bloc 5."